In [71]:
from pathlib import Path
import json
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [72]:
BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORT_DIR = BASE_DIR / "reports"

In [73]:
INPUT_FILE = Path("Year 2009-2010.csv")
print("Dataset found:", INPUT_FILE.exists())

Dataset found: True


In [74]:
READ_EXCEL = INPUT_FILE.suffix.lower() in {".xlsx", ".xls"}

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

In [75]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(REPORT_DIR / "cleaning.log"),
        logging.StreamHandler()
    ]
)

In [76]:
def load_data(file_path):
    """Load CSV or Excel data and preserve the original column names."""
    if not file_path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {file_path}\n"
            "Place your downloaded file in the data/raw folder."
        )

    if file_path.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(file_path)
    else:
        try:
            df = pd.read_csv(file_path, encoding="utf-8")
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding="latin1")

    logging.info("Loaded %s rows and %s columns", *df.shape)
    return df

In [77]:
def standardize_columns(df):
    """Map common Online Retail column names to canonical names."""
    aliases = {
    "Invoice": "order_id",
    "StockCode": "product_id",
    "Description": "product_description",
    "Quantity": "quantity",
    "InvoiceDate": "transaction_datetime",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country",
    }

     
    rename_map = {}
    for column in df.columns:
        cleaned_name = str(column).strip()
        matched = next(
            (
                original
                for original in aliases
                if original.lower() == cleaned_name.lower()
            ),
            None
        )
        if matched:
            rename_map[column] = aliases[matched]
        else:
            rename_map[column] = cleaned_name.lower().replace(" ", "_")

    df = df.rename(columns=rename_map)
    

    required = [
        "order_id",
        "product_id",
        "quantity",
        "transaction_datetime",
        "unit_price",
        "customer_id",
    ]

    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(
            f"Required columns are missing: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    return df


In [78]:
def create_quality_profile(df, stage):
    """Summarize missing values, data types, and duplicate rows."""
    profile = pd.DataFrame({
        "column": df.columns,
        "data_type": [str(df[c].dtype) for c in df.columns],
        "missing_count": [int(df[c].isna().sum()) for c in df.columns],
        "missing_percent": [
            round(df[c].isna().mean() * 100, 2) for c in df.columns
        ],
        "unique_values": [int(df[c].nunique(dropna=True)) for c in df.columns],
    })

    profile.insert(0, "stage", stage)
    return profile 

In [79]:
def clean_transactions(raw_df):
    df = raw_df.copy()

    rows_before = len(df)
    exact_duplicates = int(df.duplicated().sum())
    df = df.drop_duplicates().copy()
    text_columns = [
        "order_id",
        "product_id",
        "product_description",
        "country",
        "customer_id",
    ]

    for column in text_columns:
        if column in df.columns:
            df[column] = df[column].astype("string").str.strip()
            df[column] = df[column].replace("", pd.NA)

    for column in ["order_id", "product_id", "customer_id"]:
        if column in df.columns:
            df[column] = df[column].astype("string").str.strip()

    df["transaction_datetime"] = pd.to_datetime(
        df["transaction_datetime"], errors="coerce"
    )
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

    for column in ["product_description", "country"]:
        if column in df.columns:
            df[column] = df[column].str.replace(
                r"\s+", " ", regex=True
            )

    df["missing_customer_id"] = df["customer_id"].isna()
    df["invalid_datetime"] = df["transaction_datetime"].isna()
    df["invalid_quantity"] = df["quantity"].isna()
    df["invalid_unit_price"] = df["unit_price"].isna()

    df["is_cancellation"] = (
        df["order_id"]
        .fillna("")
        .str.upper()
        .str.startswith("C")
    )

    df["is_return_or_negative_quantity"] = df["quantity"].lt(0)
    df["is_zero_price"] = df["unit_price"].eq(0)
    df["is_negative_price"] = df["unit_price"].lt(0)
    df["line_value"] = df["quantity"] * df["unit_price"]
    invalid_required = (
        df["transaction_datetime"].isna()
        | df["quantity"].isna()
        | df["unit_price"].isna()
        | df["order_id"].isna()
        | df["product_id"].isna()
        | df["unit_price"].lt(0)
    )

    quarantine = df.loc[invalid_required].copy()
    quarantine["quarantine_reason"] = "Missing required field or negative price"

    cleaned = df.loc[~invalid_required].copy()
    for column in ["quantity", "unit_price", "line_value"]:
        values = cleaned[column].dropna()

        if len(values) > 0:
            q1 = values.quantile(0.25)
            q3 = values.quantile(0.75)
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr

            cleaned[f"{column}_outlier_flag"] = (
                (cleaned[column] < lower)
                | (cleaned[column] > upper)
            )
        else:
            cleaned[f"{column}_outlier_flag"] = False
    summary = {
        "rows_before_cleaning": int(rows_before),
        "exact_duplicate_rows_removed": exact_duplicates,
        "rows_after_exact_deduplication": int(rows_before - exact_duplicates),
        "rows_quarantined": int(len(quarantine)),
        "rows_in_cleaned_transactions": int(len(cleaned)),
        "rows_missing_customer_id_in_cleaned_data": int(
            cleaned["customer_id"].isna().sum()
        ),
        "cancellation_rows": int(cleaned["is_cancellation"].sum()),
        "negative_quantity_rows": int(
            cleaned["is_return_or_negative_quantity"].sum()
        ),
        "zero_price_rows": int(cleaned["is_zero_price"].sum()),
        "negative_price_rows_in_quarantine": int(
            df["is_negative_price"].sum()
        ),
    }

    logging.info("Cleaning summary: %s", summary)
    return cleaned, quarantine, summary


In [80]:
def create_customer_features(cleaned):
    """
    Build customer features using transactions before a fixed cutoff.
    Label future repeat purchases in a separate outcome window.

    The latest 30 days are used as the outcome window.
    This requires sufficient historical data and customer coverage.
    """
    data = cleaned.loc[cleaned["customer_id"].notna()].copy()
    data["is_completed_purchase_line"] = (
        ~data["is_cancellation"]
        & data["quantity"].gt(0)
        & data["unit_price"].gt(0)
    )

    purchases = data.loc[data["is_completed_purchase_line"]].copy()

    if purchases.empty:
        raise ValueError(
            "No qualifying purchase records were found. "
            "Review the dataset and purchase rules."
        )

    latest_date = purchases["transaction_datetime"].max()
    cutoff = latest_date.normalize() - pd.Timedelta(days=30)
    outcome_end = cutoff + pd.Timedelta(days=30)
    history = purchases.loc[
        purchases["transaction_datetime"] <= cutoff
    ].copy()
    future = purchases.loc[
        (purchases["transaction_datetime"] > cutoff)
        & (purchases["transaction_datetime"] <= outcome_end)
    ].copy()

    if history.empty:
        raise ValueError(
            "No historical purchases remain before the cutoff. "
            "Use a dataset with a longer date range or adjust the horizon."
        )

    customer_features = history.groupby("customer_id").agg(
        first_purchase_date=("transaction_datetime", "min"),
        last_purchase_date=("transaction_datetime", "max"),
        order_count=("order_id", "nunique"),
        transaction_line_count=("order_id", "size"),
        total_spend=("line_value", "sum"),
        average_line_value=("line_value", "mean"),
        distinct_products=("product_id", "nunique"),
    ).reset_index()
    order_values = (
        history.groupby(["customer_id", "order_id"])["line_value"]
        .sum()
        .reset_index(name="order_value")
    )
    average_order_values = (
        order_values.groupby("customer_id")["order_value"]
        .mean()
        .reset_index(name="average_order_value")
    )

    customer_features = customer_features.merge(
        average_order_values, on="customer_id", how="left"
    )

    customer_features["recency_days"] = (
        cutoff - customer_features["last_purchase_date"]
    ).dt.days

    customer_features["customer_tenure_days"] = (
        customer_features["last_purchase_date"]
        - customer_features["first_purchase_date"]
    ).dt.days

    future_order_counts = future.groupby("customer_id")["order_id"].nunique()

    customer_features["future_order_count"] = (
        customer_features["customer_id"]
        .map(future_order_counts)
        .fillna(0)
        .astype(int)
    )
    customer_features["repeat_purchase_label"] = (
        customer_features["future_order_count"] >= 1
    ).astype(int)
    customer_features["churn_proxy_label"] = (
        customer_features["future_order_count"] == 0
    ).astype(int)
    customer_features["observation_cutoff"] = cutoff
    customer_features["outcome_window_end"] = outcome_end

    logging.info(
        "Feature cutoff: %s; outcome window end: %s",
        cutoff, outcome_end
    )
    logging.info(
        "Customer feature table contains %s customers",
        len(customer_features)
    )

    return customer_features

In [81]:
def save_quality_plots(cleaned):
    """Save a missingness plot and numeric distribution plots."""
    if cleaned.empty:
        logging.warning("No cleaned rows available for plotting.")
        return

    missing_percent = (
        cleaned.isna().mean().mul(100).sort_values(ascending=False)
    )
    missing_percent = missing_percent[missing_percent > 0]

    if not missing_percent.empty:
        plt.figure(figsize=(10, 5))
        missing_percent.plot(kind="bar")
        plt.ylabel("Missing values (%)")
        plt.title("Missing Values by Column After Cleaning")
        plt.tight_layout()
        plt.savefig(REPORT_DIR / "missing_values.png", dpi=150)
        plt.close()

    numeric_columns = [
        c for c in ["quantity", "unit_price", "line_value"]
        if c in cleaned.columns
    ]

    for column in numeric_columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(cleaned[column].dropna(), bins=50)
        plt.title(f"Distribution of {column}")
        plt.tight_layout()
        plt.savefig(REPORT_DIR / f"{column}_distribution.png", dpi=150)
        plt.close()


In [82]:
def main():
    raw = load_data(INPUT_FILE)
    raw_profile = create_quality_profile(raw, "raw")

    standardized = standardize_columns(raw)
    cleaned, quarantine, summary = clean_transactions(standardized)

    cleaned_profile = create_quality_profile(cleaned, "cleaned")
    quality_profile = pd.concat(
        [raw_profile, cleaned_profile],
        ignore_index=True
    )

    customer_features = create_customer_features(cleaned)

    cleaned.to_csv(
        PROCESSED_DIR / "cleaned_transactions.csv", index=False
    )
    quarantine.to_csv(
        PROCESSED_DIR / "quarantined_records.csv", index=False
    )
    customer_features.to_csv(
        PROCESSED_DIR / "customer_features.csv", index=False
    )
    quality_profile.to_csv(
        REPORT_DIR / "data_quality_profile.csv", index=False
    )

    with open(REPORT_DIR / "cleaning_summary.json", "w",
              encoding="utf-8") as f:
        json.dump(summary, f, indent=4)

    save_quality_plots(cleaned)

    logging.info("Pipeline completed successfully.")
    logging.info("Processed files saved in %s", PROCESSED_DIR)
    logging.info("Reports saved in %s", REPORT_DIR)


if __name__ == "__main__":
    main()


2026-10-13 14:05:54,705 | INFO | Loaded 525461 rows and 8 columns
2026-10-13 14:05:58,538 | INFO | Cleaning summary: {'rows_before_cleaning': 525461, 'exact_duplicate_rows_removed': 6865, 'rows_after_exact_deduplication': 518596, 'rows_quarantined': 3, 'rows_in_cleaned_transactions': 518593, 'rows_missing_customer_id_in_cleaned_data': 107830, 'cancellation_rows': 10182, 'negative_quantity_rows': 12302, 'zero_price_rows': 3681, 'negative_price_rows_in_quarantine': 3}
2026-10-13 14:05:59,600 | INFO | Feature cutoff: 2010-11-09 00:00:00; outcome window end: 2010-12-09 00:00:00
2026-10-13 14:05:59,601 | INFO | Customer feature table contains 4024 customers
2026-10-13 14:06:06,650 | INFO | Pipeline completed successfully.
2026-10-13 14:06:06,652 | INFO | Processed files saved in c:\yuvaintern\week3\data\processed
2026-10-13 14:06:06,653 | INFO | Reports saved in c:\yuvaintern\week3\reports
